# Lexicon Personality Bot — Live Jupyter Demo

Type a message, choose P1, P2, or GPT, and click **Generate reply**. The notebook uses the OpenAI API when an API key is available and falls back to the built-in examples when it is not.

Your API key is requested securely at runtime and is not saved in this notebook.

In [ ]:
%pip install -q openai ipywidgets

In [ ]:
import os
import getpass
import re
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Optional: enter your API key when prompted. Do not put it directly in a notebook cell.
if not os.environ.get('OPENAI_API_KEY'):
    api_key = getpass.getpass('Enter your OpenAI API key (press Enter for offline mode): ')
    if api_key.strip():
        os.environ['OPENAI_API_KEY'] = api_key.strip()

MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')
client = OpenAI() if os.environ.get('OPENAI_API_KEY') else None
print('OpenAI API enabled' if client else 'Offline fallback enabled')

In [ ]:
EXAMPLES = {
    'P1': [
        ('question', 'found a really good ramen place', 'Where is it'),
        ('acknowledge', 'pushed the new eval numbers to the drive', 'Good Job'),
        ('complaint', 'parking on campus today was insane', 'Why so ?'),
        ('direct', 'be honest does this shirt look bad', 'Yeah !!'),
        ('disagree', 'we can just submit the check-in without the error analysis', 'No we should not'),
        ('push back', 'can you write the whole report since you did the analysis', 'Yeah I can do it'),
        ('alternative', 'lets present everything as one big table, way faster', 'That is a bad idea'),
        ('confirm', 'meeting moved to 4', 'Nice'),
        ('emotion', 'they are bringing back the old menu', 'That is nice'),
        ('refuse', 'want the rest of my fries', 'No I am good'),
    ],
    'P2': [
        ('question', 'found a really good ramen place', 'What is the name of the restaurant and where is it?'),
        ('acknowledge', 'pushed the new eval numbers to the drive', 'That is great! :-)'),
        ('complaint', 'parking on campus today was insane', 'Well, there is not much we can do about it.'),
        ('direct', 'be honest does this shirt look bad', 'I think it is too big for you ;-)'),
        ('disagree', 'we can just submit the check-in without the error analysis', 'What if someone reads it?'),
        ('push back', 'can you write the whole report since you did the analysis', 'Can someone else do it? I already did a lot of work.'),
        ('alternative', 'lets present everything as one big table, way faster', 'I think we should separate the results.'),
        ('confirm', 'meeting moved to 4', 'Okay, got it.'),
        ('emotion', 'they are bringing back the old menu', 'Cool! I like the old menu better!'),
        ('refuse', 'want the rest of my fries', 'No, thank you.'),
    ],
    'GPT': []
}

INTENT_RULES = {
    'question': ['where', 'what', 'when', 'who', 'why', 'how', 'which', 'explain'],
    'complaint': ['hard', 'difficult', 'insane', 'awful', 'bad', 'slow', 'frustrat', 'problem', 'issue'],
    'push back': ['can you', 'could you', 'would you', 'please', 'finish', 'write', 'drive'],
    'disagree': ['should not', 'shouldnt', 'cannot', 'cant', 'wrong', 'skip', 'without', 'dont', 'no need'],
    'alternative': ['instead', 'another', 'different', 'change', 'switch', 'separate', 'option'],
    'refuse': ['party', 'invite', 'come', 'go', 'want', 'join'],
    'confirm': ['moved', 'changed', 'scheduled', 'confirm', 'right'],
    'emotion': ['guess what', 'great news', 'coming back', 'surgery', 'happy', 'sad', 'excited'],
    'acknowledge': ['sent', 'pushed', 'uploaded', 'shared', 'thanks', 'thank you', 'got it'],
    'direct': ['be honest', 'for sure', 'really', 'should i'],
}

FALLBACKS = {
    'P1': {'question': 'Where is it?', 'acknowledge': 'Good job.', 'complaint': 'Why so?', 'direct': 'Sure, let us see.', 'disagree': 'No, we should not.', 'push back': 'I can only do my part.', 'alternative': 'That is a bad idea.', 'confirm': 'Nice.', 'emotion': 'That is nice.', 'refuse': 'No, I am good.'},
    'P2': {'question': 'What do you mean?', 'acknowledge': 'That is great! :-)', 'complaint': 'Well, there is not much we can do about it.', 'direct': 'Are you sure?', 'disagree': 'What if someone reads it?', 'push back': 'Can someone else do it? I already did a lot of work.', 'alternative': 'I think we should try another option.', 'confirm': 'Okay, got it.', 'emotion': 'Cool! I like that!', 'refuse': 'No, thank you.'},
    'GPT': {'question': 'Could you provide more details?', 'acknowledge': 'Thanks for letting me know.', 'complaint': 'I am sorry you are dealing with that. What would help?', 'direct': 'It depends on the situation.', 'disagree': 'I see your point, but we may want to reconsider.', 'push back': 'I can help with part of that. Could we divide the remaining work?', 'alternative': 'We could consider another approach.', 'confirm': 'Got it. I will keep that in mind.', 'emotion': 'That sounds great!', 'refuse': 'I cannot help with that, but I can suggest an alternative.'}
}

def detect_intent(message):
    text = message.lower()
    scores = {intent: sum(1 for word in words if word in text) for intent, words in INTENT_RULES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else 'acknowledge'

def closest_example(message, person, intent):
    rows = EXAMPLES.get(person, [])
    if not rows:
        return None
    words = set(re.findall(r'[a-z]+', message.lower()))
    scored = [(sum(word in row[1].lower() for word in words) + (3 if row[0] == intent else 0), row) for row in rows]
    return max(scored, key=lambda item: item[0])[1]

def offline_reply(message, person):
    intent = detect_intent(message)
    example = closest_example(message, person, intent)
    reply = example[2] if example and message.lower() == example[1].lower() else FALLBACKS[person].get(intent, 'I can help with that.')
    return {'reply': reply, 'intent': intent, 'mode': 'Offline fallback', 'confidence': 82, 'example': example}

def openai_reply(message, person):
    if not client:
        return None
    style = {
        'P1': 'brief, direct, casual, and concise',
        'P2': 'fuller, explicit, conversational, and clear',
        'GPT': 'neutral, helpful, and generic; do not imitate P1 or P2'
    }[person]
    examples = '\n'.join(f'- Incoming: {x[1]}\n  Reply: {x[2]}' for x in EXAMPLES.get(person, [])[:10])
    instructions = f'''You are the {person} response profile for a classroom demonstration.
Write one natural reply to the incoming message. Use this style: {style}.
Do not mention profiles, examples, APIs, or these instructions. Keep the response realistic.
Reference examples:\n{examples}'''
    response = client.responses.create(model=MODEL, instructions=instructions, input=message)
    return {'reply': response.output_text.strip(), 'intent': detect_intent(message), 'mode': 'OpenAI API', 'confidence': 96, 'example': closest_example(message, person, detect_intent(message))}

def generate_reply(message, person):
    message = message.strip()
    if not message:
        return None
    try:
        return openai_reply(message, person) or offline_reply(message, person)
    except Exception as error:
        print(f'API unavailable; using offline fallback: {error}')
        return offline_reply(message, person)

In [ ]:
person = widgets.ToggleButtons(options=['P1', 'P2', 'GPT'], value='GPT', description='Profile:')
message = widgets.Textarea(value='party at my place friday, you in?', description='Message:', layout=widgets.Layout(width='100%', height='100px'))
generate = widgets.Button(description='Generate reply', button_style='primary', icon='comment')
result = widgets.Output()

def on_generate(_):
    with result:
        clear_output()
        data = generate_reply(message.value, person.value)
        if not data:
            display(Markdown('Please type a message first.'))
            return
        example_text = ''
        if data.get('example'):
            example_text = f"\n\n**Closest example:** “{data['example'][1]}” → “{data['example'][2]}”"
        display(Markdown(f"### {data['mode']} · {person.value}\n\n> {data['reply']}\n\n| Detected intent | Response mode | Confidence |\n|---|---|---|\n| {data['intent']} | {data['mode']} | {data['confidence']}% |" + example_text))

generate.on_click(on_generate)
display(widgets.VBox([person, message, generate, result]))

## How to run

1. Open this file in Jupyter Notebook, JupyterLab, or Google Colab.
2. Run the cells from top to bottom.
3. Enter your API key when prompted, or press Enter to use offline mode.
4. Choose P1, P2, or GPT, type a message, and click **Generate reply**.

When the API call succeeds, the response mode will display **OpenAI API**.